<a href="https://colab.research.google.com/github/Derikklok/HPC-MPI-Question-2/blob/main/Demo_2_Q2_MPI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!sudo apt-get update
!sudo apt-get install -y openmpi-bin openmpi-common libopenmpi-dev

Hit:1 https://cli.github.com/packages stable InRelease
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [83.6 kB]
Get:9 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,201 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:11 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,509 kB]
Hit:13 https://ppa.launchpadcontent.net/ubuntugis/p

In [2]:
%%writefile program_1.c
#include <stdio.h>
#include <stdlib.h>

int main() {
    int N = 1024;

    // Allocate matrices
    int **A = (int **)malloc(N * sizeof(int *));
    int **T = (int **)malloc(N * sizeof(int *));
    for (int i = 0; i < N; i++) {
        A[i] = (int *)malloc(N * sizeof(int));
        T[i] = (int *)malloc(N * sizeof(int));
    }

    // Initialize A[i][j] = i + j
    for (int i = 0; i < N; i++)
        for (int j = 0; j < N; j++)
            A[i][j] = i + j;

    // Serial transpose
    for (int i = 0; i < N; i++)
        for (int j = 0; j < N; j++)
            T[j][i] = A[i][j];

    // Print snippet
    printf("Serial transpose snippet:\n");
    for (int i = 0; i < 5; i++) {
        for (int j = 0; j < 5; j++)
            printf("%d ", T[i][j]);
        printf("\n");
    }

    // Free memory
    for (int i = 0; i < N; i++) {
        free(A[i]);
        free(T[i]);
    }
    free(A);
    free(T);

    return 0;
}


Writing program_1.c


In [8]:
!mpicc program_1.c -o program_1
!mpirun -np 1 --allow-run-as-root --oversubscribe ./program_1

Serial transpose snippet:
0 1 2 3 4 
1 2 3 4 5 
2 3 4 5 6 
3 4 5 6 7 
4 5 6 7 8 


In [6]:
%%writefile program_2.c
#include <stdio.h>
#include <stdlib.h>
#include <mpi.h>

int main(int argc, char *argv[]) {
    MPI_Init(&argc, &argv);

    int rank, size;
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);

    int N = 1024;  // Use 2048 if needed
    int rows_per_process = N / size;

    // Root allocates full matrix A and final result T
    int *A = NULL;
    int *T = NULL;

    if (rank == 0) {
        A = (int *)malloc(N * N * sizeof(int));
        T = (int *)malloc(N * N * sizeof(int));

        // Initialize A[i][j] = i + j
        for (int i = 0; i < N; i++)
            for (int j = 0; j < N; j++)
                A[i*N + j] = i + j;
    }

    // Each process gets its chunk of rows
    int *local_A = (int *)malloc(rows_per_process * N * sizeof(int));

    MPI_Scatter(
        A, rows_per_process * N, MPI_INT,
        local_A, rows_per_process * N, MPI_INT,
        0, MPI_COMM_WORLD
    );

    // Each process computes partial transpose: T[j][i]
    // Local result size: N columns x rows_per_process rows
    int *local_T = (int *)malloc(rows_per_process * N * sizeof(int));

    for (int i = 0; i < rows_per_process; i++) {
        for (int j = 0; j < N; j++) {
            // i-th local row maps to global row = rank*rows_per_process + i
            int global_row = rank * rows_per_process + i;
            local_T[j * rows_per_process + i] = local_A[i * N + j];
        }
    }

    // Gather transposed blocks to the root process
    MPI_Gather(
        local_T, rows_per_process * N, MPI_INT,
        T,       rows_per_process * N, MPI_INT,
        0, MPI_COMM_WORLD
    );

    // Root prints the first 5x5 block of the transpose
    if (rank == 0) {
        printf("MPI Distributed Transpose snippet:\n");
        for (int i = 0; i < 5; i++) {
            for (int j = 0; j < 5; j++)
                printf("%d ", T[i*N + j]);
            printf("\n");
        }
    }

    free(local_A);
    free(local_T);
    if (rank == 0) {
        free(A);
        free(T);
    }

    MPI_Finalize();
    return 0;
}


Overwriting program_2.c


In [7]:
!mpicc program_2.c -o program_2
!mpirun -np 4 --allow-run-as-root --oversubscribe ./program_2

MPI Distributed Transpose snippet:
0 1 2 3 4 
4 5 6 7 8 
8 9 10 11 12 
12 13 14 15 16 
16 17 18 19 20 
